# Lesson 5.3 — Dynamic Baselines and Anomaly Detection

## Learning Objectives
- Understand how AppDynamics learns dynamic baselines automatically
- Explain standard deviation-based anomaly detection
- Configure baseline settings and time-of-day baselines
- Understand when to use baselines vs. static thresholds

**Reference**: [Dynamic Baselines](https://help.splunk.com/en/appdynamics-saas/application-performance-monitoring/26.2.0/business-transactions/monitor-the-business-transaction-performance/dynamic-baselines) | [Anomaly Detection](https://help.splunk.com/en/appdynamics-saas/get-started/26.2.0/alert-and-respond/anomaly-detection)

## 1. Why Static Thresholds Fail

Imagine setting a static threshold: "Alert if response time > 1000ms".

Problems:
- **Weekend vs. weekday traffic**: A 500ms response is normal at 2 AM; at 2 PM on Black Friday, normal is 200ms
- **Post-deployment changes**: A new feature legitimately increases response time from 200ms to 800ms — no problem, but you'd still get alerts
- **Different BTs have different profiles**: `/static/image.png` should be < 50ms; `/generate-report` might normally take 5 seconds
- **Seasonal patterns**: E-commerce traffic spikes every holiday season

**AppDynamics' solution**: Learn what's normal for *each metric individually* and alert on *deviation from normal*.

In [ ]:
# Demonstrate the problem with static thresholds vs dynamic baselines

import random
import statistics
import matplotlib.pyplot as plt

random.seed(42)

# Simulate a week of hourly average response times
# Pattern: busiest 9 AM - 6 PM weekdays, quiet nights and weekends
def simulate_week(base_rt=300, noise=40):
    hours = []
    for day in range(7):
        for hour in range(24):
            is_weekday = day < 5
            is_business_hours = 9 <= hour <= 18
            load_factor = 1.5 if (is_weekday and is_business_hours) else 0.7
            rt = random.gauss(base_rt * load_factor, noise * load_factor)
            hours.append(max(50, rt))
    return hours

week_rts = simulate_week(300, 40)

# Static threshold approach
STATIC_THRESHOLD = 500  # ms — a reasonable guess
static_alerts = sum(1 for rt in week_rts if rt > STATIC_THRESHOLD)

# Dynamic baseline approach (sliding window)
def compute_dynamic_baseline(values, window=168):  # 168 = 1 week of hours
    """Compute mean and std dev over all historical values."""
    mean = statistics.mean(values[:window])
    stdev = statistics.stdev(values[:window])
    return mean, stdev

baseline_mean, baseline_std = compute_dynamic_baseline(week_rts)
dynamic_critical_threshold = baseline_mean + 3 * baseline_std  # 3 sigma
dynamic_warning_threshold  = baseline_mean + 2 * baseline_std  # 2 sigma
dynamic_alerts = sum(1 for rt in week_rts if rt > dynamic_critical_threshold)

print("Static vs Dynamic Threshold Comparison:")
print(f"  Total hours analyzed    : {len(week_rts)}")
print(f"  Overall mean response   : {baseline_mean:.1f}ms")
print(f"  Std deviation           : {baseline_std:.1f}ms")
print()
print(f"  Static threshold (500ms): {static_alerts} alerts ({static_alerts/len(week_rts)*100:.1f}% of time)")
print(f"  Dynamic critical (3σ):    {dynamic_critical_threshold:.0f}ms — {dynamic_alerts} alerts ({dynamic_alerts/len(week_rts)*100:.1f}% of time)")
print(f"  Dynamic warning (2σ):     {dynamic_warning_threshold:.0f}ms")
print()
print("Insight: Static threshold can be either too noisy or too silent.")
print("Dynamic baselines self-adjust to actual traffic patterns.")

In [ ]:
# Visualize the baseline with anomaly detection

fig, ax = plt.subplots(figsize=(14, 5))

hours_x = list(range(len(week_rts)))
ax.plot(hours_x, week_rts, alpha=0.6, linewidth=1, color='steelblue', label='Avg Response Time')
ax.axhline(STATIC_THRESHOLD, color='red', linestyle='--', linewidth=1.5, label=f'Static threshold ({STATIC_THRESHOLD}ms)')
ax.axhline(dynamic_critical_threshold, color='darkred', linestyle='-', linewidth=1.5, label=f'Dynamic critical (3σ = {dynamic_critical_threshold:.0f}ms)')
ax.axhline(dynamic_warning_threshold, color='orange', linestyle='-', linewidth=1, label=f'Dynamic warning (2σ = {dynamic_warning_threshold:.0f}ms)')
ax.axhline(baseline_mean, color='green', linestyle=':', linewidth=1, label=f'Baseline mean ({baseline_mean:.0f}ms)')

# Shade business hours
for day in range(5):  # Mon-Fri
    start = day * 24 + 9
    end   = day * 24 + 18
    ax.axvspan(start, end, alpha=0.05, color='yellow')

ax.set_xlabel('Hour of week')
ax.set_ylabel('Avg Response Time (ms)')
ax.set_title('Dynamic Baselines vs Static Threshold — 1 Week')
ax.legend(fontsize=8, loc='upper right')
ax.set_xlim(0, len(week_rts))

# Mark days
days = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
for i, day in enumerate(days):
    ax.text(i * 24 + 12, ax.get_ylim()[0] + 10, day, ha='center', fontsize=8, alpha=0.7)

plt.tight_layout()
plt.savefig('dynamic_baselines.png', dpi=100, bbox_inches='tight')
plt.show()

## 2. How AppDynamics Calculates Baselines

AppDynamics uses a **rolling statistical model** per metric per BT:

1. Collects data for the first **2 weeks** (learning period)
2. Calculates **mean** and **standard deviation** for each minute-of-day and day-of-week
   - Tuesday 2:00 PM has a different baseline than Tuesday 3:00 AM
3. As new data arrives, the baseline **updates continuously** (exponential moving average)
4. Slow/Very Slow thresholds = mean ± N standard deviations (default N=3 for Very Slow)

### Baseline types:
| Baseline | Period | Best for |
|----------|--------|----------|
| **Default Baseline** | Last 2 weeks rolling | Most situations |
| **Weekly Hourly Baseline** | Same hour, same day last 8 weeks | Strong weekly patterns |
| **Custom Baseline** | You define the time window | Special events (Black Friday) |

### When to use static thresholds instead:
- SLA contracts specify exact millisecond limits
- New applications (no historical data yet)
- Compliance requirements

In [ ]:
# EXERCISE 5.3: Choose baseline vs. static threshold for each scenario

scenarios = [
    {
        "scenario": "Payment processing must complete in < 3 seconds per contractual SLA",
        "recommendation": "???",  # TODO: static or dynamic baseline?
        "reason": "???"
    },
    {
        "scenario": "Product search response time varies greatly between business hours and off-hours",
        "recommendation": "???",
        "reason": "???"
    },
    {
        "scenario": "A batch report generation job runs every night at 2 AM and takes 45 minutes normally",
        "recommendation": "???",
        "reason": "???"
    },
    {
        "scenario": "JVM heap usage — alert if unusually high compared to the past 2 weeks",
        "recommendation": "???",
        "reason": "???"
    },
    {
        "scenario": "Newly deployed microservice with 3 days of data",
        "recommendation": "???",
        "reason": "???"
    },
]

print("Baseline vs. Static Threshold Decision:")
for s in scenarios:
    print(f"\nScenario: {s['scenario']}")
    print(f"  Recommendation: {s['recommendation']}")
    print(f"  Reason: {s['reason']}")

## Summary

- AppDynamics **automatically learns** a statistical baseline (mean + std dev) for every metric over the first 2 weeks
- Slow = 3× baseline, Very Slow = 6× baseline by default (configurable per BT)
- Baselines account for **time-of-day and day-of-week** patterns automatically
- Use **static thresholds** for contractual SLAs or when baseline learning is not possible
- Use **dynamic baselines** for organic traffic patterns and relative anomaly detection

---

**Next**: [Lesson 5.4 — Alert Troubleshooting and Tuning](lesson-5.4-alert-troubleshooting.ipynb)